# Unit 5 习题课：比例数据的推断与卡方检验

本 Notebook 用于练习 Unit 5 的比例数据推断：

- 单个比例：Wald / Wilson / Wilson + continuity correction / exact binomial test
- 两个独立比例差异：比例差 CI 与 z 检验
- RC 联立表：Pearson χ²、期望频数、Cramér's V、Fisher exact test
- GOF 拟合优度：χ² goodness-of-fit test

> 使用方式：每个题目后先在“学生作答区”运行或补全代码，再查看“参考答案”。


#### 习题课规则
1. 不能用AI工具；
2. 第一轮： 课前预习完成，课堂上Canvas 独立提交 （<10分钟）；
3. 课堂分组讨论（20分钟）
4. 第二轮： Canvas再次独立提交（15分钟）
5. 从平均分最高的2组中，每组各随机挑选1名完成两轮Canvas提交的幸运同学，获赠本月底出版的教材）
6. 判断题每个问题1分，答案和理由各0.5分；
7. 选择题每个问题1分；
8. 计算题每个小问题1分。

## 0. 环境准备

如果本机缺少 `scipy` 或 `statsmodels`，可先安装：

```bash
pip install scipy statsmodels numpy
```

In [21]:
import math
import numpy as np

from scipy.stats import binomtest, chi2_contingency, fisher_exact, chisquare
from statsmodels.stats.proportion import (
    proportion_confint,
    proportions_ztest,
    confint_proportions_2indep,
)

np.set_printoptions(precision=4, suppress=True)

In [22]:
def fmt_ci(ci, digits=3):
    """Format a confidence interval tuple/list."""
    return f"[{ci[0]:.{digits}f}, {ci[1]:.{digits}f}]"


def wilson_cc_interval(x, n, alpha=0.05):
    """Wilson score interval with continuity correction.

    Formula follows the common Newcombe-style continuity-corrected Wilson interval.
    It is useful for small samples or extreme proportions, and stays within [0, 1].
    """
    if not (0 <= x <= n):
        raise ValueError("x must be between 0 and n")
    if n <= 0:
        raise ValueError("n must be positive")

    # 95% by default. For other alpha values, scipy could provide norm.ppf;
    # here we keep the course's 95% setting explicit and lightweight.
    if abs(alpha - 0.05) > 1e-12:
        from scipy.stats import norm
        z = norm.ppf(1 - alpha / 2)
    else:
        z = 1.959963984540054

    p = x / n
    z2 = z * z
    denom = 2 * (n + z2)

    if x == 0:
        lower = 0.0
    else:
        lower = (
            2 * n * p + z2 - 1
            - z * math.sqrt(z2 - 2 - 1 / n + 4 * p * (n * (1 - p) + 1))
        ) / denom

    if x == n:
        upper = 1.0
    else:
        upper = (
            2 * n * p + z2 + 1
            + z * math.sqrt(z2 + 2 - 1 / n + 4 * p * (n * (1 - p) - 1))
        ) / denom

    return max(0.0, lower), min(1.0, upper)


def cramers_v(table, chi2_stat):
    table = np.asarray(table)
    n = table.sum()
    r, c = table.shape
    return math.sqrt(chi2_stat / (n * min(r - 1, c - 1)))

---

## 一、是非判断题

判断下列说法是否正确，并用 1 句话说明理由。

1. 单个比例的 Wald 区间依赖正态近似；若样本比例接近 0 或 1，Wilson 区间通常比 Wald 区间更稳健。
2. 若比例检验得到 `p >= 0.05`，就可以证明两个总体比例完全相同。
3. 比较两个独立比例时，只要两个组各自的 95% CI 有重叠，就一定不能认为两组有差异。
4. 卡方检验的统计量越大，说明观测频数与零假设下期望频数的差距越大；p 值计算看右尾概率。
5. 2x2 联立表中如果有期望频数小于 5，只要总样本量超过 40，仍应优先使用普通 Pearson 卡方检验。

### 学生作答区

提示：每题先判断“适用条件是否满足”，再判断方法是否合适；答案建议写成“对/错 + 1 句理由”。

1. 对，当样本比例接近0/1时，在这种计算情况下，wald区间误差较大而且容易越界，Wilson区间更加合理。
2. 错，p>=0.05只能表明在显著性水平为0.95的条件下我们认为两个总体比例没有差别。
3. 错，置信区间不能进行比较，可以计算两个比例差值的置信区间来进行比较。
4. 对，根据计算卡方统计量的公式可以知道，p值计算的是卡方分布大于观测值的概率，卡方越大代表观测值与期望值之间的差距越大。
5. 错，要用Fisher's Exact Test。

---

## 二、标准化选择题（单选）

### 1. 方法选择：小样本高比例

某调查抽取 50 人，其中 45 人已接种疫苗。估计接种率 95% CI 时，最合适的做法是：

A. 直接使用 Wald 区间，因为样本量已经达到 50  
B. 优先使用 Wilson 或带连续性校正的 Wilson 区间  
C. 使用两个独立比例 z 检验  
D. 使用单因素 ANOVA  

### 2. 两个独立比例 z 检验

比较两组恢复率，X 组 200 人中 156 人恢复，Y 组 180 人中 126 人恢复。检验 `H0: p1 = p2` 时，z 检验的标准误应基于：

A. 两组各自样本比例的非合并标准误  
B. 合并比例 `p_pool = (x1 + x2) / (n1 + n2)`  
C. t 分布下的样本标准差  
D. RC 表中每格的期望频数  

### 3. 2x2 小样本联立表

某不良反应研究得到 2x2 表：新方案组 1/40 出现严重反应，常规组 7/40 出现严重反应。若某格期望频数小于 5，最合适的检验是：

A. Pearson 卡方检验，不需要任何校正  
B. Fisher 精确检验  
C. 单样本比例 z 检验  
D. 拟合优度卡方检验  

### 4. GoF 自由度

将身高分成 5 个箱，并用样本数据估计正态分布的 `mu` 和 `sigma` 后做拟合优度检验。自由度应为：

A. 5  
B. 4  
C. 2  
D. 1  

### 5. 卡方结果报告

下列哪种报告最符合本单元要求？

A. “p < .05，所以结果显著。”  
B. “两变量有关，因为百分比看起来不同。”  
C. “卡方值很大，因此效应也很大。”  
D. “χ²(2, N = 450) = 12.16, p = .002, Cramér's V = 0.16。”

### 学生作答区

提示：每题只有一个最佳答案。可以先在旁边标出关键词，例如“小样本/极端比例”“合并比例”“E < 5”“df = k - 1 - m”“APA 报告要素”。

1. B，未接种数量小于10，所以要使用Wilson
2. B，的确需要合并比例
3. B，期望频数小于5，采用Fishe's Exact Test
4. C，需要估计的参数有两个，所以自由度为5-3=2
5. D，非常完整的APA汇报格式

---

## 三、案例综合计算题：校园流感防控项目

某高校在 2026 春季开展流感疫苗接种与健康教育项目。请根据以下 4 组数据完成统计推断。结果需包含统计量、p 值、置信区间或期望频数，并解释是否满足适用条件。

## A. 单个比例：小样本且比例接近 1

预调查随机抽取 50 名学生，其中 45 人已接种流感疫苗。

请完成：

1. 计算样本比例 `p_hat`。
2. 检查正态近似条件：`n * p_hat` 与 `n * (1 - p_hat)` 是否都大于 10？
3. 分别计算 Wald、Wilson、Wilson + continuity correction 区间，并说明哪一个更适合作为主要报告。
4. 若学校目标是接种率高于 80%，请用 exact binomial test 或谨慎使用 z 检验进行单侧检验，并说明为什么不能机械套用 Wald/z 方法。

In [36]:
# A. 学生作答区：单个比例
# 数据：50 名学生中 45 人已接种
x, n = 45, 50

# 提示 1：样本比例
p_hat = x / n
print("样本比例为", p_hat)

# 提示 2：检查正态近似条件
success_count = n * p_hat
failure_count = n * (1 - p_hat)
# 常用经验条件：success_count > 10 且 failure_count > 10
print("阳性样本数为", round(success_count), "大于10")
print("阴性样本数为", round(failure_count), "小于10")

print("---")

# 提示 3：单比例置信区间
from statsmodels.stats.proportion import proportion_confint
# Wald 区间：method="normal"
# Wilson 区间：method="wilson"
ci_wald = proportion_confint(count=x, nobs=n, method="normal")
print("流感疫苗接种率的95% Wald 置信区间是 :",fmt_ci(ci_wald))
ci_wilson = proportion_confint(count=x, nobs=n, method="wilson")
print("流感疫苗接种率的95% Wilson 置信区间是 :",fmt_ci(ci_wilson))
ci_wilson_cc = wilson_cc_interval(x, n)
print("流感疫苗接种率的95% Wilson + continuity correction 置信区间是 :",fmt_ci(ci_wilson_cc))
print("Wilson + continuity correction 置信区间最为合适：由于该样本阴性样本数小于10，所以不适合采用Wald区间来进行计算；加上连续性校正后的Wilson区间能修正离散偏差，最为合适。")
print("---")
# 提示 4：检验学校目标 p0 = 0.80，备择假设为 p > 0.80
exact = binomtest(x, n, p=0.80, alternative="greater")
z, p_z = proportions_ztest(x, n, value=0.80,
                            alternative="larger",
                           prop_var=0.80)

# 提示 5：输出时可以使用 fmt_ci(ci)
print("p_hat =", p_hat)
print("Wald CI =", fmt_ci(ci_wald))
print("Wilson CI =", fmt_ci(ci_wilson))
print("Exact p =", exact.pvalue.round(3))
print("因为该样本阴性数小于10，不满足近似正态分布的条件，所以不能套用Wald/z方法，计算得到p=.048<.05，拒绝零假设，认为接种率高于80%")

样本比例为 0.9
阳性样本数为 45 大于10
阴性样本数为 5 小于10
---
流感疫苗接种率的95% Wald 置信区间是 : [0.817, 0.983]
流感疫苗接种率的95% Wilson 置信区间是 : [0.786, 0.957]
流感疫苗接种率的95% Wilson + continuity correction 置信区间是 : [0.774, 0.963]
Wilson + continuity correction 置信区间最为合适：由于该样本阴性样本数小于10，所以不适合采用Wald区间来进行计算；加上连续性校正后的Wilson区间能修正离散偏差，最为合适。
---
p_hat = 0.9
Wald CI = [0.817, 0.983]
Wilson CI = [0.786, 0.957]
Exact p = 0.048
因为该样本阴性数小于10，不满足近似正态分布的条件，所以不能套用Wald/z方法，计算得到p=.048<.05，拒绝零假设，认为接种率高于80%


## B. 两个独立比例差异

健康教育项目中，提醒组 200 人中 156 人完成接种；对照组 180 人中 126 人完成接种。

请完成：

1. 计算两组接种率和比例差 `p1 - p2`。
2. 检查每组“接种/未接种”人数是否足够大。
3. 计算两组各自的 Wilson 95% CI。
4. 计算比例差的 95% CI，并执行两个独立比例 z 检验。
5. 解释：是否可以说提醒组接种率显著更高？

In [45]:
# B. 学生作答区：两个独立比例差异
# 数据：提醒组 156/200，对照组 126/180
x1, n1 = 156, 200
x2, n2 = 126, 180

# 提示 1：计算两组比例和比例差
p1 = x1 / n1
p2 = x2 / n2
diff = p1 - p2
print("提醒组接种率为：", round(p1, 3))
print("对照组接种率为：", round(p2, 3))
print("两组接种率之差为：", round(diff, 3))
print("---")

# 提示 2：检查每组接种/未接种人数
group1_counts = (x1, n1 - x1)
group2_counts = (x2, n2 - x2)
# 常用经验条件：每组成功数和失败数都不要太小，例如都 >= 10
print("提醒组和对照组两组接种和未接种人数均大于10")

# 提示 3：两组各自 Wilson 置信区间
ci1 = proportion_confint(x1, n1, method="wilson")
ci2 = proportion_confint(x2, n2, method="wilson")
print("提醒组接种率的 Wilson 95% 置信区间为：", fmt_ci(ci1))
print("对照组接种率的 Wilson 95% 置信区间为：", fmt_ci(ci2))
print("---")

# 提示 4：比例差的置信区间，compare="diff" 表示 p1 - p2
ci_diff = confint_proportions_2indep(
    x1, n1, x2, n2,
    method="wald",
    compare="diff" )
print("两组接种率的差值的 Wald 95% 置信区间为：", fmt_ci(ci_diff))

# 提示 5：两个独立比例 z 检验；检验 H0: p1 = p2 时函数会使用合并比例
z, p = proportions_ztest([x1, x2], [n1, n2])
print("两个比例差值z检验的z统计量为：", z.round(3))
print("两个比例差值z检验的p值为：", p.round(3))
print("可以看到，两个独立比例的z检验z=1.78，p=.075>.05，不拒绝原假设；95%CI=[-0.008, 0.168]，包含0。均说明两组接种率无明显差别。")

# 提示 6：解释时看 p 值是否 < 0.05，以及比例差 CI 是否包含 0

提醒组接种率为： 0.78
对照组接种率为： 0.7
两组接种率之差为： 0.08
---
提醒组和对照组两组接种和未接种人数均大于10
提醒组接种率的 Wilson 95% 置信区间为： [0.718, 0.832]
对照组接种率的 Wilson 95% 置信区间为： [0.629, 0.762]
---
两组接种率的差值的 Wald 95% 置信区间为： [-0.008, 0.168]
两个比例差值z检验的z统计量为： 1.78
两个比例差值z检验的p值为： 0.075
可以看到，两个独立比例的z检验z=1.78，p=.075>.05，不拒绝原假设；95%CI=[-0.008, 0.168]，包含0。均说明两组接种率无明显差别。


## C. RC 联立表卡方检验与 Fisher 边界

研究者比较三种提醒方式与“是否完成接种”的关系：

| 接种状态 | 无提醒 | 短信提醒 | 短信+电话 |
|---|---:|---:|---:|
| 完成接种 | 60 | 72 | 90 |
| 未完成接种 | 90 | 78 | 60 |

请完成：

1. 这是几个分类变量？每个变量有几个水平？
2. 写出零假设与备择假设。
3. 用 `chi2_contingency` 计算 χ²、df、p 值和期望频数矩阵。
4. 检查期望频数是否满足卡方检验条件。
5. 计算 Cramér's V，并按 APA 格式报告。

In [65]:
# C1. 学生作答区：RC 联立表卡方检验
# 行：完成接种 / 未完成接种
# 列：无提醒 / 短信提醒 / 短信+电话
print("一共有两个分类变量，其中“是否完成接种”有两个水平，“提醒方式”有三个水平")

table = np.array([[60, 72, 90],
                  [90, 78, 60]])

# 提示 1：先确认 table 是 2 x 3 联立表
table.shape
print("零假设：是否完成接种与提醒方式相互独立")
print("备选假设：是否完成接种与提醒方式有关联")
print("---")
# 提示 2：Pearson 卡方独立性检验
# correction=False：本题不是 2 x 2 表，不做 Yates 校正
chi2, p, df, expected = chi2_contingency(table, correction=False)
print("卡方检验的卡方统计量为：", chi2.round(3))
print("卡方检验的自由度为：", df)
print("卡方检验的p值为：", p.round(3))
print("---")
# 提示 3：检查期望频数
print(expected)
print("最小期望数为：", np.min(expected))
print("所有期望数是否大于5：", np.all(expected >= 5))
print("所有期望数是否大于10：", np.all(expected >= 10))

# 提示 4：Cramér's V
N = table.sum()
r, c = table.shape
V = np.sqrt(chi2 / (N * min(r - 1, c - 1)))
# 或者直接使用前面定义的函数：V = cramers_v(table, chi2)
print("Cramer's V为：", V.round(3))
print("---")

# 提示 5：APA 格式报告需要 chi2、df、N、p、Cramér's V
# print(f"χ²({df}, N = {N}) = {chi2:.2f}, p = {p:.3f}, Cramér's V = {V:.2f}")
print(f"χ²({df}, N = {N}) = {chi2:.2f}, p = {p:.3f}, Cramér's V = {V:.2f}")
print("p=.002<.05，拒绝零假设，认为是否完成接种与提醒方式有关联；V=0.16，表明两个因素之间存在弱关联")

一共有两个分类变量，其中“是否完成接种”有两个水平，“提醒方式”有三个水平
零假设：是否完成接种与提醒方式相互独立
备选假设：是否完成接种与提醒方式有关联
---
卡方检验的卡方统计量为： 12.162
卡方检验的自由度为： 2
卡方检验的p值为： 0.002
---
[[74. 74. 74.]
 [76. 76. 76.]]
最小期望数为： 74.0
所有期望数是否大于5： True
所有期望数是否大于10： True
Cramer's V为： 0.164
---
χ²(2, N = 450) = 12.16, p = 0.002, Cramér's V = 0.16
p=.002<.05，拒绝零假设，认为是否完成接种与提醒方式有关联；V=0.16，表明两个因素之间存在弱关联


### C2. 小样本 2x2 表：Fisher Exact Test

| 方案 | 严重不良反应 | 无严重不良反应 |
|---|---:|---:|
| 新方案 | 1 | 39 |
| 常规方案 | 7 | 33 |

请完成：

1. 先计算期望频数。
2. 若有格子 `E < 5`，应选择普通 χ²、Yates 校正，还是 Fisher 精确检验？
3. 用 `fisher_exact` 计算双侧 p 值，并解释结论。

In [ ]:
# C2. 学生作答区：小样本 2 x 2 表与 Fisher Exact Test
# 行：新方案 / 常规方案
# 列：严重不良反应 / 无严重不良反应
rare = np.array([[1, 39],
                 [7, 33]])

# 提示 1：先用 chi2_contingency 查看期望频数，不急着报告普通卡方结果
chi2_small, p_small, df_small, expected_small = chi2_contingency(
    rare, correction=False
)
print(expected_small)
print("可以看到有两个期望值小于5，采用Fisher精确检验")

# 提示 2：若存在 E < 5，本题主报告 Fisher 精确检验
# np.any(expected_small < 5)

# 提示 3：Fisher 精确检验调用格式
oddsratio, p_fisher = fisher_exact(rare, alternative="two-sided")
print("OR值为：", oddsratio.round(3))
print("Fisher检验得到的p值为：", p_fisher.round(3))

# 提示 4：可以顺手比较 Yates 校正，但要说明主报告为什么选择 Fisher
chi2_yates, p_yates, _, _ = chi2_contingency(rare, correction=True)
print("卡方统计量为：", chi2_yates.round(3))
print("Yates校正得到的p值为：", p_yates.round(3))
print("由于有两个期望值不仅小于10，而且还小于5，所以单纯的Yates校正还不够，必须使用Fisher检验。得到p=.057>.05，说明不拒绝原假设，认为两个因素之间有关联；oddsratio值为0.121，说明新方案发生严重不良反应的风险更低")

[[ 4. 36.]
 [ 4. 36.]]
可以看到有两个期望值小于5，采用Fisher精确检验
OR值为： 0.121
Fisher检验得到的p值为： 0.057
卡方统计量为： 3.472
Yates校正得到的p值为： 0.062
由于有两个期望值不仅小于10，而且还小于5，所以单纯的Yates校正还不够，必须使用Fisher检验。得到p=.057>.05，说明不拒绝原假设，认为两个因素之间有关联；OR值为0.121，说明新方案发生严重不良反应的风险更低


## D. GOF 拟合优度卡方检验

某校 200 名学生血型分布如下。当地总体参考比例为 A: 43%, B: 40%, O: 12%, AB: 5%。

| 血型 | A | B | O | AB |
|---|---:|---:|---:|---:|
| 观测频数 O | 88 | 70 | 32 | 10 |
| 理论比例 | 0.43 | 0.40 | 0.12 | 0.05 |

请完成：

1. 写出 GoF 的零假设。
2. 计算期望频数 `E = n * p`，并检查是否都不小于 5。
3. 用 `chisquare` 完成拟合优度检验。
4. 给出自由度，并解释是否有证据认为本校血型分布不同于当地总体参考分布。

In [ ]:
# D. 学生作答区：GOF 拟合优度卡方检验
# 观测频数：A, B, O, AB
O = np.array([88, 70, 32, 10])

# 理论比例：A, B, O, AB
p0 = np.array([0.43, 0.40, 0.12, 0.05])

# 提示 1：检查理论比例是否加和为 1
print(p0.sum())

# 提示 2：根据理论比例计算期望频数
E = O.sum() * p0
print(E)
 
# 提示 3：检查 GOF 卡方检验的期望频数条件
print("期望频数是否都不小于5：", np.all(E >= 5))

# 提示 4：拟合优度检验
chi2_gof, p_gof = chisquare(f_obs=O, f_exp=E)
print("卡方统计量为：", chi2_gof.round(3))
print("p值为：", p_gof.round(3))

# 提示 5：自由度
# 本题参考比例是外部给定的，没有从样本估计参数，所以 m = 0
df = len(O) - 1
print("自由度为：", df)

# 提示 6：解释时看 p 值是否 < 0.05；不能拒绝 H0 不等于证明 H0 完全正确
print("卡方统计的总数为200，自由度为3，统计量为3.963，p=.265>.05，说明不拒绝原假设，认为本校血型分布符合当地总体参考分布")


1.0
[86. 80. 24. 10.]
期望频数是否都不小于5： True
卡方统计量为： 3.963
p值为： 0.265
自由度为： 3
卡方统计的总数为200，自由度为3，统计量为3.963，p=.265>.05，说明不拒绝原假设，认为本校血型分布符合当地总体参考分布


---

## 四、汇总：本单元方法选择速查

| 场景 | 常用方法 | 关键条件 / 注意点 |
|---|---|---|
| 单个比例估计 | Wilson CI | 小样本或 p 接近 0/1 时优先 Wilson 或 Wilson + continuity correction |
| 单个比例检验 | exact binomial test 或 z test | 小样本/极端比例时 exact 更稳健；z test 要检查近似条件 |
| 两个独立比例差异 | two-proportion z test；比例差 CI | 两组成功/失败人数都应足够大；检验 H0: p1=p2 时用合并比例 |
| 2x2 小样本表 | Fisher exact test | 若期望频数过小，避免机械使用普通 Pearson χ² |
| R x C 联立表 | Pearson χ² independence test | 检查期望频数；必要时合并稀疏类别 |
| GoF | chisquare | 期望频数建议均 >= 5；df = k - 1 - m |

常见报告要素：样本量、估计值/效应量、CI、统计量、df、p 值、方法适用条件。